# lum-model-vision — standalone model demo

Every model in the `lum_vision` package, driven from a single frame. No database,
no Redis, no Celery, no `config.yaml` — this notebook is the proof that the models
are genuinely decoupled from the SmartOffice application.

Each section takes a frame in, draws its result, displays it, and writes a PNG to
`notebooks/output/`.

| Section | Model | Needs |
|---|---|---|
| 1 | `PersonDetector` | nothing |
| 2 | `FaceDetector` | nothing |
| 3 | `PersonTracker` | the **boxmot fork** (see setup) |
| 4 | `FaceMatcher` | nothing |
| 5 | `GlobalTrackManager` | ReID weights (auto-downloaded) |
| 6 | `ActionRecognizer` | a running Ollama server |

## 0. Setup

`lum_vision` now lives in its own private repo — install it before running this
notebook:

```
pip install "lum-model-vision @ git+ssh://git@github.com/lumiohub-ai/lum-model-vision@v0.1.0"
```

**PersonTracker needs the boxmot fork.** Upstream boxmot on PyPI lacks the
`custom_features` argument BoT-SORT is called with here, so section 3 will raise a
clear error telling you to install:

```
pip install 'boxmot @ git+https://github.com/humblebeeintel/yolo_tracking@5a2b2a63b59aa26e118b4950a4932be51f8b8de3'
```

In [ ]:
import sys, warnings
from pathlib import Path

warnings.filterwarnings("ignore")

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

import cv2
import numpy as np
import matplotlib.pyplot as plt

import lum_vision
from lum_vision import (
    VisionConfig, ModelFactory, InMemoryEmbeddingProvider, PersonTracker,
)

OUT = REPO / "notebooks" / "output"
OUT.mkdir(parents=True, exist_ok=True)

import boxmot, torch
print(f"lum_vision {lum_vision.__version__}")
print(f"python    {sys.executable}")
print(f"torch     {torch.__version__} | cuda available: {torch.cuda.is_available()}")
print(f"boxmot    {'site-packages' if 'site-packages' in boxmot.__file__ else boxmot.__file__}")
print(f"output    {OUT}")

In [ ]:
# Model weights live under model_cache_dir. The default (~/.cache/lum-vision) is
# writable by you; ./volumes/models is root-owned on this host, so do NOT point
# there from a notebook or downloads will fail with Permission denied.
config = VisionConfig()

print(f"model_cache_dir : {config.model_cache_dir}")
print(f"  weights       : {config.weights_dir}")
print(f"  insightface   : {config.insightface_dir}")
print(f"match_threshold : {config.match_threshold}")

models = ModelFactory(config)  # nothing loads until first access

### Pick an input frame

Point `INPUT` at any image, or at a video to grab a frame from.

In [ ]:
INPUT = REPO / "cam2.mp4"   # an image path works too
FRAME_NUMBER = 0            # ignored for images


def load_frame(path, frame_number=0):
    """Read a BGR frame from an image or a video."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)

    if path.suffix.lower() in {".mp4", ".avi", ".mov", ".mkv"}:
        cap = cv2.VideoCapture(str(path))
        if frame_number:
            cap.set(cv2.CAP_PROP_POS_FRAMES, frame_number)
        ok, frame = cap.read()
        cap.release()
        if not ok:
            raise RuntimeError(f"Could not read frame {frame_number} from {path}")
        return frame

    frame = cv2.imread(str(path))
    if frame is None:
        raise RuntimeError(f"Could not decode image {path}")
    return frame


def show(image_bgr, title, save_as=None, width=14):
    """Display a BGR image inline and optionally save it to notebooks/output/."""
    h, w = image_bgr.shape[:2]
    plt.figure(figsize=(width, width * h / w))
    plt.imshow(cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis("off")
    plt.show()

    if save_as:
        dest = OUT / save_as
        cv2.imwrite(str(dest), image_bgr)
        print(f"saved -> {dest}")


def draw_box(img, xyxy, label, color, thickness=2):
    """Draw a labelled box with a filled caption strip above it."""
    x1, y1, x2, y2 = (int(v) for v in xyxy)
    cv2.rectangle(img, (x1, y1), (x2, y2), color, thickness)
    if not label:
        return
    (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
    cv2.rectangle(img, (x1, max(0, y1 - th - 6)), (x1 + tw + 4, y1), color, -1)
    cv2.putText(img, label, (x1 + 2, max(10, y1 - 4)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)


frame = load_frame(INPUT, FRAME_NUMBER)
show(frame, f"input — {Path(INPUT).name} {frame.shape[1]}x{frame.shape[0]}", "00_input.png")

## 1. Person detection — `PersonDetector`

First access downloads `yolo26s.pt` (~20 MB) into `model_cache_dir/weights`.

Note the package exposes the underlying ultralytics model as `.model` — there is no
`detect()` wrapper on `PersonDetector` today, so we filter for class 0 (person) here.

In [ ]:
result = models.person_detector.model(frame, verbose=False)[0]

persons = [
    {"bbox": box, "confidence": float(conf)}
    for box, conf, cls in zip(
        result.boxes.xyxy.tolist(),
        result.boxes.conf.tolist(),
        result.boxes.cls.tolist(),
    )
    if int(cls) == 0 and conf >= config.person_detection_threshold
]

canvas = frame.copy()
for i, p in enumerate(persons):
    draw_box(canvas, p["bbox"], f"person {i} {p['confidence']:.2f}", (0, 200, 0))

print(f"{len(persons)} person(s) at threshold {config.person_detection_threshold}")
show(canvas, f"1. PersonDetector — {len(persons)} persons", "01_person_detection.png")

## 2. Face detection — `FaceDetector`

Returns a bbox, a 512-d normalized embedding, and 5 landmarks per face. First access
downloads the `buffalo_l` model set into `model_cache_dir/insightface`.

If onnxruntime cannot load CUDA it falls back to CPU — slower, but it still works.

In [ ]:
faces = models.face_detector.extract_face_features(frame)

canvas = frame.copy()
for i, f in enumerate(faces):
    x1, y1, x2, y2, conf, _ = f["bbox"]
    draw_box(canvas, (x1, y1, x2, y2), f"face {i} {conf:.2f}", (0, 165, 255))
    for (lx, ly) in f["landmarks"]:
        cv2.circle(canvas, (int(lx), int(ly)), 2, (255, 0, 255), -1)

print(f"{len(faces)} face(s)")
if faces:
    e = faces[0]["embedding"]
    print(f"embedding: shape={e.shape} norm={np.linalg.norm(e):.4f} (pre-normalized)")

show(canvas, f"2. FaceDetector — {len(faces)} faces + landmarks", "02_face_detection.png")

## 3. Person tracking — `PersonTracker`

Tracking needs more than one frame, so this runs over a short span of the video and
draws the track IDs on the last frame. IDs persisting across frames is the thing to
look for.

Raises a `RuntimeError` if you have upstream boxmot rather than the fork — that is
deliberate, because the fallback path silently tracks worse.

In [ ]:
N_FRAMES = 30

tracker = PersonTracker(camera_id=0, with_reid=False, confidence_threshold=0.5)

cap = cv2.VideoCapture(str(INPUT))
last_frame, active = None, []

for _ in range(N_FRAMES):
    ok, f = cap.read()
    if not ok:
        break

    det = models.person_detector.model(f, verbose=False)[0]
    dets = [
        {"bbox": b, "confidence": float(c), "keypoints": None}
        for b, c, k in zip(det.boxes.xyxy.tolist(), det.boxes.conf.tolist(), det.boxes.cls.tolist())
        if int(k) == 0 and c >= config.person_detection_threshold
    ]
    active, _removed = tracker.update(dets, frame=f)
    last_frame = f

cap.release()


def id_color(track_id):
    """Stable, well-separated colour per track ID."""
    hue = int((track_id * 47) % 180)
    bgr = cv2.cvtColor(np.uint8([[[hue, 200, 255]]]), cv2.COLOR_HSV2BGR)[0][0]
    return int(bgr[0]), int(bgr[1]), int(bgr[2])


canvas = last_frame.copy()
for t in active:
    tid = t.get("track_id", t.get("id"))
    draw_box(canvas, t["bbox"], f"ID {tid}", id_color(int(tid)), thickness=3)

print(f"after {N_FRAMES} frames: {len(active)} active tracks")
print(f"track IDs: {sorted(int(t.get('track_id', t.get('id'))) for t in active)}")
show(canvas, f"3. PersonTracker — frame {N_FRAMES}, {len(active)} tracks", "03_person_tracking.png")

## 4. Face matching — `FaceMatcher`

This is the model that used to open its own Postgres connection. It now reads through
the `EmbeddingProvider` protocol, so any source works — here an in-memory gallery
built from the faces found in section 2.

In the real app the provider is `PgVectorStore`, passed in unchanged.

In [ ]:
if not faces:
    print("No faces found in section 2 — pick a frame with visible faces.")
else:
    gallery_names = [f"person_{i}" for i in range(len(faces))]
    gallery_embs = np.array([f["embedding"] for f in faces], dtype=np.float32)

    matcher = ModelFactory(
        config,
        embedding_provider=InMemoryEmbeddingProvider(gallery_names, gallery_embs),
    ).face_matcher

    canvas = frame.copy()
    for i, f in enumerate(faces):
        query = f["embedding"].reshape(1, -1)
        idx, score = matcher.get_best_match(matcher.compute_similarities(query))
        name = matcher.db_names[idx]
        hit = score >= matcher.match_threshold

        x1, y1, x2, y2, _, _ = f["bbox"]
        draw_box(canvas, (x1, y1, x2, y2),
                 f"{name} {score:.2f}" if hit else f"unknown {score:.2f}",
                 (0, 200, 0) if hit else (0, 0, 255))

    print(f"gallery of {len(gallery_names)}; each face matched against it")
    print("(scores are ~1.0 because every face is matched against itself)")
    show(canvas, "4. FaceMatcher — identity + similarity", "04_face_matching.png")

## 5. Cross-camera ReID — `GlobalTrackManager`

Extracts an appearance embedding per person crop so the same individual can be linked
across cameras. The config comes from the YAML packaged inside `lum_vision`, which is
why this works from any working directory.

First run downloads the OSNet weights (~9 MB) into `model_cache_dir/weights`.

In [ ]:
gtm = models.global_track_manager

print(f"enabled          : {gtm.enabled}")
print(f"similarity thresh: {gtm.similarity_threshold}")
print(f"reid model       : {gtm.reid_model_name}")
print(f"reid weights     : {gtm.reid_weights_path}")
print(f"weights on disk  : {Path(gtm.reid_weights_path).exists()}")
print(f"device           : {gtm.reid_device}  (auto-detected, not hardcoded)")

In [ ]:
# Compare two person crops by appearance. Same person -> high similarity.
if len(persons) < 2:
    print("Need at least 2 detected persons to compare.")
else:
    crops = []
    for p in persons[:4]:
        x1, y1, x2, y2 = (int(v) for v in p["bbox"])
        crops.append(frame[max(0, y1):y2, max(0, x1):x2])

    embs = []
    for c in crops:
        e = gtm._extract_body_embedding(c) if hasattr(gtm, "_extract_body_embedding") else None
        embs.append(e)

    if any(e is None for e in embs):
        print("ReID embedding unavailable (weights missing or model disabled).")
    else:
        M = np.array(embs, dtype=np.float32)
        M /= np.linalg.norm(M, axis=1, keepdims=True)
        sim = M @ M.T
        print("pairwise appearance similarity (diagonal = self = 1.0):")
        print(np.round(sim, 3))

    strip = np.hstack([cv2.resize(c, (128, 256)) for c in crops])
    show(strip, "5. GlobalTrackManager — person crops compared", "05_reid_crops.png", width=8)

## 6. Action recognition — `ActionRecognizer`

The only model needing an external service: a running Ollama with a vision model.

It is now **synchronous** — `recognize()` blocks and returns. The queue and worker
threads that used to live inside it are the application's job
(`src/pipeline/action_worker.py`), so nothing is started behind your back here.

Actions are config-driven; `VisionConfig()` has none by default, so we supply them.

In [ ]:
from lum_vision import ActionConfig, ActionRecognizer

action_config = ActionConfig(
    enabled=True,
    ollama_api_url="http://localhost:11534",  # so-face-oybek-ollama-1, mapped from container's 11434
    model_name="gemma3:4b",
    # Ollama unloads the model after 5 min idle; a cold load (weights + CUDA graph)
    # took ~28s for a trivial text prompt alone, so 30s left no margin for a real
    # vision request. A canceled load also leaves the model unloaded for the next
    # call too, so a too-short timeout compounds into repeated cold starts.
    inference_timeout=60,
    actions={
        "sleeping":              {"backend_type": "sleeping",    "description": "head resting on desk, eyes closed"},
        "using phone":           {"backend_type": "phone_usage", "description": "holding a phone, looking down at a device"},
        "working with computer": {"backend_type": "working",     "description": "sitting at a desk facing a screen or typing"},
        "talking with someone":  {"backend_type": "talking",     "description": "facing another person, gesturing"},
        "idle":                  {"backend_type": "unknown",     "description": "standing or sitting without a clear activity"},
    },
)

recognizer = ActionRecognizer(action_config)
print(f"{len(recognizer.actions_config)} actions configured")
print(f"prompt preview:\n{recognizer.prompt_template[:220]}...")

In [ ]:
if not persons:
    print("No person crops to classify.")
else:
    x1, y1, x2, y2 = (int(v) for v in persons[0]["bbox"])
    crop = frame[max(0, y1):y2, max(0, x1):x2]

    result = recognizer.recognize(crop, metadata={"track_id": 0})

    if result is None:
        print("Inference failed — is Ollama running at", action_config.ollama_api_url, "?")
        print("Start it with:  docker compose up ollama    (or: ollama serve)")
        label = "no result"
    else:
        print(f"action        : {result.action}")
        print(f"activity_type : {result.activity_type}")
        print(f"raw output    : {result.raw_output!r}")
        print(f"inference time: {result.inference_time:.2f}s")
        label = f"{result.action or 'unknown'}"

    canvas = frame.copy()
    draw_box(canvas, (x1, y1, x2, y2), label, (255, 100, 0), thickness=3)
    show(canvas, f"6. ActionRecognizer — {label}", "06_action_recognition.png")

## 7. Everything at once

Persons, faces and identities on one frame — the combination the pipeline actually runs.

In [ ]:
canvas = frame.copy()

for i, p in enumerate(persons):
    draw_box(canvas, p["bbox"], f"person {i} {p['confidence']:.2f}", (0, 200, 0))

for i, f in enumerate(faces):
    x1, y1, x2, y2, conf, _ = f["bbox"]
    draw_box(canvas, (x1, y1, x2, y2), f"face {i}", (0, 165, 255))

summary = f"{len(persons)} persons | {len(faces)} faces"
cv2.rectangle(canvas, (0, 0), (canvas.shape[1], 34), (0, 0, 0), -1)
# ASCII only here: cv2.putText cannot render non-ASCII glyphs (they draw as "???")
cv2.putText(canvas, f"lum_vision {lum_vision.__version__} | {summary}", (10, 23),
            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2, cv2.LINE_AA)

show(canvas, f"7. Combined - {summary}", "07_combined.png")

print("\nWritten to", OUT)
for f in sorted(OUT.glob("*.png")):
    print(" ", f.name)

---

## What this demonstrates

Everything above ran without Postgres, Redis, Celery, GCS, or `configs/config.yaml`.
Before the refactor none of it was possible in a notebook:

- `FaceMatcher` opened its own Postgres connection on construction
- `PersonTracker` resolved boxmot through a `sys.path` hack relative to the repo
- `GlobalTrackManager` looked for `configs/global_tracking.yaml` relative to the CWD
- `ActionRecognizer` started worker threads and fired Celery tasks on import path
- weights landed in three unrelated places, one of them the process CWD

The models now take their inputs as arguments and hand results back.